In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 


In [0]:
silver_table = "frauddetection.silver.transactions"
bornze_table = "frauddetection.bronze.transactions"
gold_table_user_behavior = "frauddetection.gold.user_transactions_behaviour"

In [0]:
silver_df = spark.read.table(bornze_table)



In [0]:
silver_df = silver_df.dropDuplicates(
    subset = ["txn_id"]
)

In [0]:
silver_df = silver_df.withColumn(
    "transaction_id",col("txn_id")
).withColumn(
    "customer_id",col("user_id")
)

In [0]:
silver_df = silver_df.select(
    col("transaction_id"),
    col("customer_id"),
    col("amount"),
    col("timestamp"),
    col("location"),
    col("ingestTimeStamp"),
    col("ingestFileName")
)


In [0]:
silver_df = silver_df.withColumn(
    "amount", round(col("amount"), 2)
)


In [0]:
silver_df = silver_df.withColumn(
    "timestamp", col("timestamp").cast("timestamp")
)

In [0]:
silver_df.limit(5).display()

In [0]:
silver_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(silver_table)

In [0]:
%sql
select * from frauddetection.silver.transactions limit 10